# Optimized VGG-BLSTM Network for Captcha Recognition

Run this cell first to unzip the dataset and install dependencies on Colab.

In [ ]:
!unzip -q cig_ps.zip -d CIG_PS_AIML/
!pip install jiwer


## 1. Imports and Data Preparation

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.notebook import tqdm
import jiwer

DATA_DIR = 'CIG_PS_AIML/cig_ps'
TRAIN_LABELS = os.path.join(DATA_DIR, 'train-labels.csv')
TRAIN_DIR = os.path.join(DATA_DIR, 'train_images')

df = pd.read_csv(TRAIN_LABELS)

# Vocabulary creation
all_chars = set()
for text in df['text'].values:
    all_chars.update(list(str(text)))

vocab = sorted(list(all_chars))
char2idx = {c: i + 1 for i, c in enumerate(vocab)}
idx2char = {i + 1: c for i, c in enumerate(vocab)}
idx2char[0] = '-' # CTC Blank Token
num_classes = len(vocab) + 1

print(f"Vocabulary Size: {len(vocab)}")


## 2. Dataset and Augmentation
We apply Random Affine transforms (rotation, shear) to prevent overfitting.

In [ ]:
class CaptchaDataset(Dataset):
    def __init__(self, df, img_dir, char2idx, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.char2idx = char2idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['image'])
        image = Image.open(img_path).convert('L') # Convert to grayscale
        if self.transform:
            image = self.transform(image)
        
        text = str(row['text'])
        target = [self.char2idx[c] for c in text]
        return image, torch.tensor(target, dtype=torch.long), row['image']

# Train with Data Augmentation
train_transform = transforms.Compose([
    transforms.Resize((32, 128)),
    transforms.RandomAffine(degrees=5, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Validation without Augmentation
val_transform = transforms.Compose([
    transforms.Resize((32, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Use 90% for training, 10% for validation
train_size = int(0.9 * len(df))
train_df = df.iloc[:train_size]
val_df = df.iloc[train_size:]

train_dataset = CaptchaDataset(train_df, TRAIN_DIR, char2idx, transform=train_transform)
val_dataset = CaptchaDataset(val_df, TRAIN_DIR, char2idx, transform=val_transform)

def collate_fn(batch):
    images, targets, paths = zip(*batch)
    images = torch.stack(images)
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    targets = torch.cat(targets)
    return images, targets, target_lengths, paths

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)


## 3. The CRNN (VGG + BLSTM) Architecture
This is a standard, deep CRNN. The CNN outputs a sequence of features that is fed directly into a 2-layer Bidirectional LSTM.

In [ ]:
class CRNN(nn.Module):
    def __init__(self, num_classes, hidden_size=256):
        super(CRNN, self).__init__()
        
        # Deep VGG-Style CNN
        self.cnn = nn.Sequential(
            # Input: 1 x 32 x 128
            nn.Conv2d(1, 64, kernel_size=3, padding=1), nn.ReLU(True),
            nn.MaxPool2d(2, 2), # 64 x 16 x 64
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(True),
            nn.MaxPool2d(2, 2), # 128 x 8 x 32
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.ReLU(True),
            nn.MaxPool2d((2, 1), stride=(2, 1)), # 256 x 4 x 32
            
            nn.Conv2d(256, 512, kernel_size=3, padding=1), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.MaxPool2d((2, 1), stride=(2, 1)), # 512 x 2 x 32
            
            nn.Conv2d(512, 512, kernel_size=2, stride=1, padding=0), nn.BatchNorm2d(512), nn.ReLU(True)
            # Output: 512 x 1 x 31
        )
        
        # 2-Layer Bidirectional LSTM
        self.rnn = nn.LSTM(512, hidden_size, bidirectional=True, num_layers=2, batch_first=True, dropout=0.25)
        
        # Classifier
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        conv_out = self.cnn(x)
        b, c, h, w = conv_out.size()
        
        assert h == 1, "Height must be 1 after CNN"
        conv_out = conv_out.squeeze(2) # [b, c, w]
        conv_out = conv_out.permute(0, 2, 1) # [b, w, c] = [Batch, SeqLen, Features]
        
        rnn_out, _ = self.rnn(conv_out)
        out = self.fc(rnn_out)
        return nn.functional.log_softmax(out, dim=2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CRNN(num_classes).to(device)
print(f"Using device: {device}")


## 4. Training Loop
We use an Adam optimizer with `ReduceLROnPlateau` and Gradient Clipping to ensure smooth convergence.

In [ ]:
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)

epochs = 50 
best_loss = float('inf')

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    
    # Wrap train_loader in tqdm for a progress bar
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
    for images, targets, target_lengths, _ in pbar:
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model(images) # [b, w, c]
        outputs = outputs.permute(1, 0, 2) # [w, b, c] required for CTCLoss

        # CNN outputs sequence length 31
        input_lengths = torch.full(size=(images.size(0),), fill_value=outputs.size(0), dtype=torch.long).to(device)

        loss = criterion(outputs, targets, input_lengths, target_lengths)
        loss.backward()
        
        # Gradient clipping prevents the model from exploding early on
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
        
        optimizer.step()
        epoch_loss += loss.item()
        
        pbar.set_postfix({'loss': loss.item()})
        
    avg_loss = epoch_loss / len(train_loader)
    print(f'Epoch {epoch+1} Average Training Loss: {avg_loss:.4f}')
    
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for images, targets, target_lengths, _ in val_loader:
            images = images.to(device)
            targets = targets.to(device)
            outputs = model(images)
            outputs = outputs.permute(1, 0, 2)
            input_lengths = torch.full(size=(images.size(0),), fill_value=outputs.size(0), dtype=torch.long).to(device)
            loss = criterion(outputs, targets, input_lengths, target_lengths)
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(val_loader)
    print(f'Epoch {epoch+1} Average Validation Loss: {avg_val_loss:.4f}')
    
    # Save the best model
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        print("--> Saved Best Model!")
        
    scheduler.step(avg_val_loss)


## 5. Calculate CER on Validation Set

In [ ]:
model.load_state_dict(torch.load('best_model.pth', map_location=device) if os.path.exists('best_model.pth') else model.state_dict())
model.eval()

all_preds = []
all_truths = []

with torch.no_grad():
    for images, targets, target_lengths, _ in tqdm(val_loader, desc='Calculating CER'):
        images = images.to(device)
        outputs = model(images) # [b, w, c]
        
        # Greedy decoding
        _, preds = outputs.max(2) # [b, w]
        preds = preds.cpu().numpy()
        
        idx = 0
        for i in range(len(target_lengths)):
            length = target_lengths[i].item()
            truth_chars = [idx2char[t.item()] for t in targets[idx:idx+length]]
            all_truths.append(''.join(truth_chars))
            idx += length
            
            pred_chars = []
            for j in range(len(preds[i])):
                if preds[i][j] != 0 and (not (j > 0 and preds[i][j - 1] == preds[i][j])):
                    pred_chars.append(idx2char[preds[i][j]])
            all_preds.append(''.join(pred_chars))

cer = jiwer.cer(all_truths, all_preds)
print(f"\n🏆 Final Best Validation CER: {cer:.4f} 🏆")


## 6. Generate `submission.csv`

In [ ]:
TEST_DIR = os.path.join(DATA_DIR, 'test_images')
test_images = sorted(os.listdir(TEST_DIR))

# Create dummy dataframe for test dataset
test_df = pd.DataFrame({'image': test_images, 'text': ['' for _ in range(len(test_images))]})
test_dataset = CaptchaDataset(test_df, TEST_DIR, char2idx, transform=val_transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

submission_results = []

model.eval()
with torch.no_grad():
    for images, _, _, paths in tqdm(test_loader, desc='Predicting Test Set'):
        images = images.to(device)
        outputs = model(images)
        
        _, preds = outputs.max(2)
        preds = preds.cpu().numpy()
        
        for i in range(len(paths)):
            pred_chars = []
            for j in range(len(preds[i])):
                if preds[i][j] != 0 and (not (j > 0 and preds[i][j - 1] == preds[i][j])):
                    pred_chars.append(idx2char[preds[i][j]])
            submission_results.append({'image': paths[i], 'text': ''.join(pred_chars)})

submission_df = pd.DataFrame(submission_results)
submission_df.to_csv('submission.csv', index=False)
print("Saved submission.csv!")
